# Fine-Tune Qwen3.5-2B for Brain (Free Colab T4 GPU)

**Before running:** Go to `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU` and click **Save**.

This notebook:
1. Installs Unsloth (fast 4-bit QLoRA)
2. Uploads your `train_data.jsonl`
3. Fine-tunes `Qwen/Qwen3.5-2B`
4. Exports a `q4_k_m` `.gguf` file for Ollama and downloads it to your PC.

In [ ]:
# Cell 1: Verify GPU & Install Unsloth
!nvidia-smi
%%capture
!pip install -U unsloth

In [ ]:
# Cell 2: Upload train_data.jsonl from your computer
from google.colab import files
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print(f"Loaded dataset: {DATASET_PATH}")

In [ ]:
# Cell 3: Load Qwen3.5-2B in 4-bit & attach LoRA adapters
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-2B",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Cell 4: Format dataset with Qwen chat template
from datasets import load_dataset

def format_prompts(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(format_prompts, batched=True)
print(f"Training examples: {len(dataset)}")
print("Sample:\n", dataset[0]["text"][:400])

In [ ]:
# Cell 5: Fine-tune with SFTTrainer
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("Training complete!", trainer_stats.metrics)

In [ ]:
# Cell 6: Quick sanity test on the 3 benchmark failure cases
FastLanguageModel.for_inference(model)

test_prompts = [
    ("You are Brain, a precise, honest local AI assistant.", "List three common programming languages. Do NOT use numbers, bullet points, or commas."),
    ("You are the Brain Autonomous Task Planner. Decompose the user goal into an ordered, compact JSON plan.", "User Goal: Search the web for Linux kernel updates and take a screenshot of the results"),
    ("You are Brain, a precise, honest local AI assistant.", "What happened on Mars today?"),
]

for sys_p, usr_p in test_prompts:
    msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": usr_p}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.2)
    print("=== TEST:", usr_p)
    print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
    print()

In [ ]:
# Cell 7: Export merged model to GGUF (q4_k_m) for Ollama & download
import glob
from google.colab import files

model.save_pretrained_gguf("qwen35_2b_brain_gguf", tokenizer, quantization_method="q4_k_m")

gguf_files = glob.glob("qwen35_2b_brain_gguf/*.gguf") or glob.glob("*.gguf")
print("Exported GGUF files:", gguf_files)
if gguf_files:
    files.download(gguf_files[0])